# Notebook Requirements
Run this code to execute the notebook if you didn't already cloned the repo.

In [ ]:
!git clone https://github.com/GiuseppeDaddario/Computer-Vision.git --recurse-submodules
%cd Computer-Vision

# Imports

In [ ]:
%pip install ultralytics --quiet
%pip install -U gdown

In [4]:
import os
os.environ['WANDB_MODE'] = 'disabled'
from pathlib import Path
from tqdm import tqdm
import shutil
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
#from src import YOLOv5_training, YOLOv5_inference #Importing from the yolov5 repo

# Globals

In [ ]:
DATASET_PATH = "dataset/CCPD2019"
DATASET_PATH_YOLO = "dataset/CCPD2019_YOLO"

# YOLOv5 paths
TRAINING_PATH_YOLO = "dataset/ccpd_2019.yaml"
test_dir = "ccpd_challenge"
TEST_PATH_YOLOV5 = f"dataset/CCPD_YOLO/{test_dir}/images/test" 

#TODO Lore: write your paths
TRAINING_PATH_PDLPR = ""
TEST_PATH_PDLPR = ""

IMG_WIDTH = 1160
IMG_HEIGHT = 720
CLASS_ID = 0 

# Utils

## Baseline

In [ ]:
##################### UTILS FOR THE BASELINE #####################

##################################################################

## YOLOv5

In [8]:
######################## UTILS FOR YOLOV5 ########################
# [Data Pre-processing] Building the bounding box in YOLO format
def convert_bbox(x1, y1, x2, y2):
    x_center = (x1 + x2) / 2.0 / IMG_WIDTH
    y_center = (y1 + y2) / 2.0 / IMG_HEIGHT
    width = abs(x2 - x1) / IMG_WIDTH
    height = abs(y2 - y1) / IMG_HEIGHT
    return x_center, y_center, width, height

# [Data Pre-processing] Extracting meta-data from the image file name
def parse_filename(fname):
    parts = fname.split('-')
    if len(parts) < 4:
        return None
    bbox_part = parts[2]
    try:
        x1y1, x2y2 = bbox_part.split('_')
        x1, y1 = map(int, x1y1.split('&'))
        x2, y2 = map(int, x2y2.split('&'))
        return convert_bbox(x1, y1, x2, y2)
    except:
        return None

# [Data Pre-processing] Processing the list of names in a .txt file
def prepare_yolo_dataset(src, dest_root="dataset", split="train", debug=False):
    images_src = f"dataset/CCPD2019/{src}"
    images_src = Path(images_src)
    dest_root = f"dataset/{dest_root}/{src}"
    images_dest = Path(dest_root) / "images" / split
    labels_dest = Path(dest_root) / "labels" / split

    os.makedirs(images_dest, exist_ok=True)
    os.makedirs(labels_dest, exist_ok=True)

    image_files = list(images_src.glob("*.jpg"))[:5] if debug else list(images_src.glob("*.jpg")) 

    for img_path in tqdm(image_files, desc=f"Processing {split} set"):
        bbox = parse_filename(img_path.name)
        if bbox is None:
            continue

        # Copy image
        dest_img = images_dest / img_path.name
        shutil.copy(img_path, dest_img)

        # Build label
        label_path = labels_dest / (img_path.stem + ".txt")
        with open(label_path, 'w') as f:
            f.write(f"{CLASS_ID} {' '.join(f'{x:.6f}' for x in bbox)}\n")

    print(f"Saved to {images_dest} and {labels_dest}")

##################################################################

## PDLPR

In [ ]:
######################## UTILS FOR PDLPR #########################
#TODO Lore: move this functions directly in the notebook
from src import PDLPR_training, PDLPR_inference
##################################################################

# Data

In [ ]:
SO="MacOs"
# Installing pixz for faster unxipping
if SO=="Linux":
    !apt-get update
    !apt-get install pixz
elif SO=="MacOs":
    !brew install pixz

zsh:1: command not found: apt-get


In [ ]:
# Downloading the .tar dataset and extracting it
%cd dataset
!gdown --id 1HDyFIuH65kVLtsXqxLA8gs0gJr7CRynh
!tar -I 'pixz -d' -xf CCPD2019.tar.xz

In [ ]:
#TODO Lore: move here the class for the dataset so that can be used in the every part of the code

## Baseline

## YOLOv5

In [10]:
prepare_yolo_dataset("ccpd_challenge", dest_root="CCPD_YOLO", split="test")

Processing test set: 100%|██████████| 5/5 [00:00<00:00, 620.50it/s]

Saved to dataset/CCPD_YOLO/ccpd_challenge/images/test and dataset/CCPD_YOLO/ccpd_challenge/labels/test


In [ ]:
# Building the .txt files, one for each image, with meta-data information in YOLO format
# Needs to be revisited for Leonardo
subsets = [
        "ccpd_base", "ccpd_blur", "ccpd_challenge", "ccpd_db",
        "ccpd_fn", "ccpd_np", "ccpd_rotate", "ccpd_tilt", "ccpd_weather"
]
splits = ["train", "val", "test"]
dest_root = "ccpd2019_yolo"

tasks = [(subset, dest_root, split) for subset in subsets for split in splits]

    # Usa un numero di processi pari ai core disponibili o meno se vuoi evitare overload
with ProcessPoolExecutor(max_workers=multiprocessing.cpu_count()) as executor:
    executor.map(prepare_yolo_dataset, tasks)


## PDLRP

In [ ]:
#TODO Lore: move here the actual preprocessing needed (the function calls)

# Network

## Baseline

## YOLOv5

## PDLRP

In [ ]:
#TODO Lore: move here the model architecture

# Train

## Baseline

## YOLOv5

In [ ]:
# Train YOLOv5s
YOLOv5_training(
    weights="yolov5s.pt",
    data=DATASET_PATH_YOLO,
    epochs=300,
    batch_size=50,
    imgsz=640,
    optimizer="Adam",
    lr0=1e-3,
    lrf=1e-5,
    cos_lr=True,
    project="runs/train",
    name="lp_detection",
    cache="ram"
)

## PDLRP

In [ ]:
# ------ training ------ #
#TODO Lore: check this works
print("PDLPR Training ...")
PDLPR_training(TRAINING_PATH_PDLPR, batch_size=32, num_epochs=3)

# Evaluation

## Baseline

## YOLOv5

In [ ]:
YOLOv5_inference(
    weights="yolov5/runs/train/exp/weights/best.pt",
    source=TEST_PATH_YOLOV5,  # cartella con almeno 5 immagini
    imgsz=640,
    device="cuda:0",  # o "cpu"
    project="runs/detect",
    name="lp_test",
    exist_ok=True
)

## PDLRP

In [ ]:
# ------ inference ------ #
#TODO Lore: check this works
print("PDLPR Inference ...")
PDLPR_inference(TEST_PATH_PDLPR, batch_size=64)